# Sequence to Sequence 翻译项目：英语-日语

## 深度学习项目启动配置

In [1]:
import matplotlib as mpl
import matplotlib.pyplot as plt
# 绘图结果直接显示在笔记本文档中，而不是弹出一个独立的窗口
%matplotlib inline
import matplotlib as mpl
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
import sklearn
import pandas as pd
import os
import sys
from tqdm.auto import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F

# 环境版本检查与硬件检测
print(sys.version_info)
for module in mpl, np, pd, sklearn, torch:
    print(module.__name__, module.__version__)

device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")
print(device)

# 随机种子设置保证可复现性
seed = 42
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
np.random.seed(seed)

sys.version_info(major=3, minor=12, micro=2, releaselevel='final', serial=0)
matplotlib 3.9.0
numpy 1.26.4
pandas 2.2.2
sklearn 1.6.0
torch 2.5.1+cu121
cuda:0


## 数据加载

In [2]:
from janome.tokenizer import Tokenizer # 导入日语分词工具
import unicodedata
import re

# 初始化日语分词器
tokenizer_ja = Tokenizer()

def preprocess_ja(w):
    """
    预处理日语句子，去除首尾空格，分词，清洗非日语/非必要字符，合并多余空格
    """
    # 1. 去除首尾空格
    w = w.strip()
    # 2. 分词：将 "本を借りてもいいですか？" 变为 "本 を 借り て も いい です か ？"
    # Janome 会自动处理日语中的全角标点
    # 使用 wakati 模式进行分词，这样可以保留标点符号
    tokens = tokenizer_ja.tokenize(w, wakati=True)
    w = " ".join(tokens)
    
    # 3. 清洗非日语/非必要字符（可选）
    # 这里保留汉字 (\u4e00-\u9fa5)、平假名 (\u3040-\u309f)、片假名 (\u30a0-\u30ff)
    # 以及常见的全角/半角标点
    w = re.sub(r"[^ \u4e00-\u9fa5\u3040-\u309f\u30a0-\u30ff?.!,。！？]+", " ", w)
    
    # 4. 合并多余空格
    w = re.sub(r'[" "]+', " ", w).strip()
    return w

def preprocess_en(w):
    """
    预处理英语句子，去除首尾空格，清洗非英语/非必要字符，合并多余空格
    """
    # 1. 转小写并去除首尾空格
    w = w.lower().strip()
    # 2. 在标点符号前后加空格
    w = re.sub(r"([?.!,])", r" \1 ", w)
    # 3. 除了字母和基本标点，其余替换为空格
    w = re.sub(r"[^a-zA-Z?.!,]+", " ", w)
    # 4. 合并多余空格
    w = re.sub(r'[" "]+', " ", w).strip()
    return w

In [ ]:
# 测试样本
en_sample = u"May I borrow this book?"
ja_sample = u"本を借りてもいいですか？"

print(preprocess_en(en_sample))
print(preprocess_ja(ja_sample))

may i borrow this book ?
本 を 借り て も いい です か ？


In [4]:
!wc -l data_en-ja/jpn.txt

'wc' �����ڲ����ⲿ���Ҳ���ǿ����еĳ���
���������ļ���


In [ ]:
# 统计数据行数
file_path = 'data_en_ja/jpn.txt'

try:
    with open(file_path, 'r', encoding='utf-8') as f:
        # 使用生成器高效统计行数，不会占用过多内存
        line_count = sum(1 for line in f)
        print(f"文件 {file_path} 总行数: {line_count}")
except FileNotFoundError:
    print(f"错误：未找到文件，请检查路径是否正确。")
except Exception as e:
    print(f"发生错误: {e}")

文件 data_en_ja/jpn.txt 总行数: 117022


## Dataset

In [6]:
from pathlib import Path
from torch.utils.data import Dataset, DataLoader

class LangPairDataset(Dataset):
    """
    自定义数据集类，用于处理语言对数据
    """
    fpath = Path(r"./data_en_ja/jpn.txt")  #数据文件路径
    cache_path = Path(r"./.cache/lang_pair.npy")  #缓存文件路径
    split_index = np.random.choice(a=["train", "test"], replace=True, p=[0.9, 0.1], size=line_count)  #按照9:1随机抽取划分训练集和测试集

    def __init__(self, mode="train", cache=False):
        if cache or not self.cache_path.exists():  #如果没有缓存，或者缓存不存在，就处理一下数据
            self.cache_path.parent.mkdir(parents=True, exist_ok=True)  #创建缓存文件夹，如果存在就忽略
            with open(self.fpath, "r", encoding="utf8") as file:
                lines = file.readlines()  # 读取数据
                lang_pair = []
                for line in lines:  #处理数据，变成list((trg, src))的形式
                    en, ja, _ = line.split('\t')
                    lang_pair.append([preprocess_en(en), preprocess_ja(ja)])
                trg, src = zip(*lang_pair)  #分离出目标语言和源语言,如果要让英语作为src，把这里交换位置即可
                trg=np.array(trg) #转换为numpy数组
                src=np.array(src) #转换为numpy数组
                np.save(self.cache_path, {"trg": trg, "src": src})  #保存为npy文件,方便下次直接读取,不用再处理
        else:
            lang_pair = np.load(self.cache_path, allow_pickle=True).item() #读取npy文件，allow_pickle=True允许读取字典
            trg = lang_pair["trg"]
            src = lang_pair["src"]

        self.trg = trg[self.split_index == mode] #按照index拿到训练集的 标签语言 --英语
        self.src = src[self.split_index == mode] #按照index拿到训练集的源语言 --日语

    def __getitem__(self, index):
        return self.src[index], self.trg[index]

    def __len__(self):
        return len(self.src)


train_ds = LangPairDataset("train")
test_ds = LangPairDataset("test")

In [7]:
# 查看数据集处理结果
print(len(train_ds))
print(len(test_ds))
print("训练集最后一个样本：\n", "source: {}\ntarget: {}".format(*train_ds[-1]))
print("测试集最后一个样本：\n", "source: {}\ntarget: {}".format(*test_ds[-1]))

105512
11510
训练集最后一个样本：
 source: 彼氏 の 友達 と 飲み に 行っ たら 彼 に 激怒 さ れ ちゃっ た その 友達 って 男 ？ 女 ？ 男 に 決まっ てる でしょ 。 どうして 彼氏 の 女 友達 と 飲み に 行か なきゃ いけ ない の ？ そりゃ そう だ 彼 ね トム って 言う ん だ けど めっちゃ イケ てる の 。 また 行き たい な
target: i went drinking with one of my boyfriend s friends , and now he s furious at me . was this friend a guy or a girl ? a guy , obviously . why would i go drinking with his female friends ? yeah , you re right . his name is tom . he s really hot , and i really want to go drinking with him again .
测试集最后一个样本：
 source: アンケート に お答え いただい た 方 の 中 から 毎月 抽選 で 名 様 に 万 円 分 の 商品 券 を プレゼント いたし ます 。
target: each month , a gift certificate worth , yen will be given to thirty people chosen at random who have completed this questionnaire .


## 构建词表

In [ ]:
from collections import Counter

# 设置特殊token ID
pad_idx = 0
bos_idx = 1
unk_idx = 2
eos_idx = 3

def get_word_idx(ds, mode="src", threshold=2):
    """
    词表创建
    """
    # 添加特殊 token
    word2idx = {
        "[PAD]": pad_idx,     # 填充 token
        "[BOS]": bos_idx,     # begin of sentence
        "[UNK]": unk_idx,     # 未知 token
        "[EOS]": eos_idx,     # end of sentence
    }
    idx2word = {value: key for key, value in word2idx.items()}
    
    index = len(idx2word)
    threshold = 1  # 出现次数低于此的token舍弃
    #太大不要都放入到内存，避免爆掉，word_list是训练集所有的词，变为一个列表，列表里每个元素都是一个词
    word_list = " ".join([pair[0 if mode=="src" else 1] for pair in ds]).split()
    # print(type(word_list))

    # 统计词频,counter类似字典，key是单词，value是出现次数
    counter = Counter(word_list) 
    # print("word count:", len(counter))

    for token, count in counter.items():
        if count >= threshold:  # 出现次数大于阈值的token加入词表
            word2idx[token] = index  # 加入词表
            idx2word[index] = token  # 加入反向词表
            index += 1

    return word2idx, idx2word

src_word2idx, src_idx2word = get_word_idx(train_ds, "src")  #源语言词表  日语
trg_word2idx, trg_idx2word = get_word_idx(train_ds, "trg")  #目标语言词表 英语



<class 'list'>
word count: 18179
<class 'list'>
word count: 11213


In [9]:
# 查看正反向词表长度是否相等
print(len(src_word2idx))
print(len(src_idx2word))
print(len(trg_word2idx))
print(len(trg_idx2word))

18183
18183
11217
11217


## Tokenizer

这里有两种处理方式，分别对应着 encoder 和 decoder 的 word embedding 是否共享，这里实现不共享的方案。

In [ ]:
class Tokenizer:
    def __init__(self, word2idx, idx2word, max_length=500, pad_idx=pad_idx, bos_idx=bos_idx, eos_idx=eos_idx, unk_idx=unk_idx):
        self.word2idx = word2idx
        self.idx2word = idx2word
        self.max_length = max_length
        self.pad_idx = pad_idx
        self.bos_idx = bos_idx
        self.eos_idx = eos_idx
        self.unk_idx = unk_idx

    def encode(self, text_list, padding_first=False, add_bos=True, add_eos=True, return_mask=False):
        """
        text_list是一个二维列表，里边的每一个元素是一个样本，样本里边是词的列表
        如果padding_first == True，则padding加载前面，否则加载后面
        return_mask: 是否返回mask(掩码），mask用于指示哪些是padding的，哪些是真实的token
        """
        max_length = min(self.max_length, add_eos + add_bos + max([len(text) for text in text_list]))
        indices_list = []
        for text in text_list:
            indices = [self.word2idx.get(word, self.unk_idx) for word in text[:max_length - add_bos - add_eos]] #如果词表中没有这个词，就用unk_idx代替，indices是一个list,里面是每个词的index,也就是一个样本的index
            if add_bos:
                indices = [self.bos_idx] + indices #添加开头
            if add_eos:
                indices = indices + [self.eos_idx] #添加结尾
            if padding_first:#padding加载前面，超参可以调
                indices = [self.pad_idx] * (max_length - len(indices)) + indices #padding加载前面
            else:#padding加载后面
                indices = indices + [self.pad_idx] * (max_length - len(indices)) #padding加载后面
            indices_list.append(indices)
        input_ids = torch.tensor(indices_list) #转换为tensor
        masks = (input_ids == self.pad_idx).to(dtype=torch.int64) #mask是一个和input_ids一样大小的tensor，0代表token，1代表padding，mask用于去除padding的影响
        return input_ids if not return_mask else (input_ids, masks)


    def decode(self, indices_list, remove_bos=True, remove_eos=True, remove_pad=True, split=False):
        """
        indices_list是一个二维列表，里边的每一个元素是一个样本，样本里边是词的index
        remove_bos: 是否去除开头
        remove_eos: 是否去除结尾
        remove_pad: 是否去除padding
        split: 是否把列表中的单词拼接，变为一个句子
        """
        text_list = []
        for indices in indices_list:
            text = []
            for index in indices:
                word = self.idx2word.get(index, "[UNK]") #如果词表中没有这个词，就用unk_idx代替
                if remove_bos and word == "[BOS]":
                    continue
                if remove_eos and word == "[EOS]":#如果到达eos，就结束
                    break
                if remove_pad and word == "[PAD]":#如果到达pad，就结束
                    break
                text.append(word) #单词添加到列表中
            #把列表中的单词拼接，变为一个句子,split=True时，返回列表，split=False时，返回句子，返回列表的目的是为了画注意力的热力图 
            text_list.append(" ".join(text) if not split else text) 
        return text_list

#两个toknizer 相对于1个toknizer的好处是embedding的参数量减少
src_tokenizer = Tokenizer(word2idx=src_word2idx, idx2word=src_idx2word) #源语言tokenizer
trg_tokenizer = Tokenizer(word2idx=trg_word2idx, idx2word=trg_idx2word) #目标语言tokenizer

In [ ]:
# trg_tokenizer.encode([["hello"], ["hello", "world"]], add_bos=True, add_eos=False,return_mask=True)
raw_text = ["hello world".split(), "Well play it by ear".split(), "this is a test".split()]
indices,mask = trg_tokenizer.encode(raw_text, padding_first=False, add_bos=True, add_eos=True,return_mask=True)

print("raw text"+'-'*10)
for raw in raw_text:
    print(raw)
print("mask"+'-'*10)
for m in mask:
    print(m)
print("indices"+'-'*10)
for index in indices:
    print(index)

raw text----------
['hello', 'world']
['Well', 'play', 'it', 'by', 'ear']
['this', 'is', 'a', 'test']
mask----------
tensor([0, 0, 0, 0, 1, 1, 1])
tensor([0, 0, 0, 0, 0, 0, 0])
tensor([0, 0, 0, 0, 0, 0, 1])
indices----------
tensor([   1,   20, 2079,    3,    0,    0,    0])
tensor([   1,    2,  425,   33, 1046, 4205,    3])
tensor([   1,  124,  209,  114, 1964,    3,    0])


## DataLoader

In [12]:
def collate_fct(batch):
    src_words = [pair[0].split() for pair in batch] #取batch内第0列进行分词，赋给src_words
    trg_words = [pair[1].split() for pair in batch] #取batch内第1列进行分词，赋给trg_words

    # [PAD] [BOS] src [EOS]
    encoder_inputs, encoder_inputs_mask = src_tokenizer.encode(
        src_words, padding_first=True, add_bos=True, add_eos=True, return_mask=True
        )

    # [BOS] trg [PAD]
    decoder_inputs = trg_tokenizer.encode(
        trg_words, padding_first=False, add_bos=True, add_eos=False, return_mask=False,
        )

    # trg [EOS] [PAD]
    decoder_labels, decoder_labels_mask = trg_tokenizer.encode(
        trg_words, padding_first=False, add_bos=False, add_eos=True, return_mask=True
        )

    return {
        "encoder_inputs": encoder_inputs.to(device=device),
        "encoder_inputs_mask": encoder_inputs_mask.to(device=device),
        "decoder_inputs": decoder_inputs.to(device=device),
        "decoder_labels": decoder_labels.to(device=device),
        "decoder_labels_mask": decoder_labels_mask.to(device=device), #mask用于去除padding的影响，计算loss时用
    } #当返回的tensor较多时，用dict返回比较合理


In [13]:
sample_dl = DataLoader(train_ds, batch_size=2, shuffle=True, collate_fn=collate_fct)

#两次执行这个代码效果不一样，因为每次执行都会shuffle
for batch in sample_dl:
    for key, value in batch.items():
        print(key)
        print(value)
        print('-'*50)
    break

encoder_inputs
tensor([[   0,    0,    0,    0,    0,    0,    0,    1, 4185,   74,  720,   68,
           16,    3],
        [   1, 2734,   74,  155,   68,   76,  351, 2612,  133,   59,   25,   42,
            5,    3]], device='cuda:0')
--------------------------------------------------
encoder_inputs_mask
tensor([[1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], device='cuda:0')
--------------------------------------------------
decoder_inputs
tensor([[   1,  325,   97,  391,  272, 2336,    9,    0,    0,    0],
        [   1,  348,  892, 1697,   26,  122,  165,   46,  240,    5]],
       device='cuda:0')
--------------------------------------------------
decoder_labels
tensor([[ 325,   97,  391,  272, 2336,    9,    3,    0,    0,    0],
        [ 348,  892, 1697,   26,  122,  165,   46,  240,    5,    3]],
       device='cuda:0')
--------------------------------------------------
decoder_labels_mask
tensor([[0, 0, 0, 0, 0, 0, 0, 1, 1,

## 模型定义

In [15]:
class Encoder(nn.Module):
    def __init__(
        self,
        vocab_size,
        embedding_dim=256,
        hidden_dim=1024,
        num_layers=1,
        ):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.gru = nn.GRU(embedding_dim, hidden_dim, num_layers=num_layers, batch_first=True)

    def forward(self, encoder_inputs):
        # encoder_inputs.shape = [batch size, sequence length]
        # bs, seq_len = encoder_inputs.shape
        embeds = self.embedding(encoder_inputs)
        # embeds.shape = [batch size, sequence length, embedding_dim]->[batch size, sequence length, hidden_dim]
        seq_output, hidden = self.gru(embeds)
        # seq_output.shape = [batch size, sequence length, hidden_dim]，hidden.shape [ num_layers, batch size, hidden_dim]
        return seq_output, hidden

In [16]:
#把上面的Encoder写一个例子，看看输出的shape
encoder = Encoder(vocab_size=100, embedding_dim=256, hidden_dim=1024, num_layers=4)
encoder_inputs = torch.randint(0, 100, (2, 50))
encoder_outputs, hidden = encoder(encoder_inputs)
print(encoder_outputs.shape)
print(hidden.shape)
print(encoder_outputs[:,-1,:])
print(hidden[-1,:,:]) #取最后一层的hidden

torch.Size([2, 50, 1024])
torch.Size([4, 2, 1024])
tensor([[-0.0636,  0.0092,  0.0103,  ..., -0.0153, -0.0004, -0.0341],
        [-0.0175,  0.0283,  0.0105,  ...,  0.0142,  0.0282, -0.0413]],
       grad_fn=<SliceBackward0>)
tensor([[-0.0636,  0.0092,  0.0103,  ..., -0.0153, -0.0004, -0.0341],
        [-0.0175,  0.0283,  0.0105,  ...,  0.0142,  0.0282, -0.0413]],
       grad_fn=<SliceBackward0>)


### BahdanauAttention公式
`score = FC(tanh(FC(EO) + FC(H)))` 

>FC(EO)的FC是Wk,FC(H)的FC是Wq,最外面的FC是V 

`attention_weights = softmax(score, axis = 1)`  

`context = sum(attention_weights * EO, axis = 1)` 

>对EO做加权求和，得到上下文向量

In [17]:
class BahdanauAttention(nn.Module):
    def __init__(self, hidden_dim=1024):
        super().__init__()
        self.Wk = nn.Linear(hidden_dim, hidden_dim) #对keys做运算，encoder的输出EO
        self.Wq = nn.Linear(hidden_dim, hidden_dim) #对query做运算，decoder的隐藏状态
        self.V = nn.Linear(hidden_dim, 1) #V矩阵

    def forward(self, query, keys, values, attn_mask=None):
        """
        正向传播
        :param query: hidden state，是decoder的隐藏状态，shape = [batch size, hidden_dim]
        :param keys: EO  [batch size, sequence length, hidden_dim]
        :param values: EO  [batch size, sequence length, hidden_dim]
        :param attn_mask:[batch size, sequence length],这里是encoder_inputs_mask
        :return:
        """
        # query.shape = [batch size, hidden_dim] -->通过unsqueeze(-2)增加维度 [batch size, 1, hidden_dim]
        # keys.shape = [batch size, sequence length, hidden_dim]
        # values.shape = [batch size, sequence length, hidden_dim]
        scores = self.V(F.tanh(self.Wk(keys) + self.Wq(query.unsqueeze(-2)))) #unsqueeze(-2)增加维度
        # score.shape = [batch size, sequence length, 1]
        if attn_mask is not None: #这个mask是encoder_inputs_mask，用来mask掉padding的部分,让padding部分socres为0
            # attn_mask is a matrix of 0/1 element,
            # 1 means to mask logits while 0 means do nothing
            # here we add -inf to the element while mask == 1
            attn_mask = (attn_mask.unsqueeze(-1)) * -1e16 #在最后增加一个维度，[batch size, sequence length] --> [batch size, sequence length, 1]
            scores += attn_mask
        scores = F.softmax(scores, dim=-2) #对每一个词的score做softmax
        # score.shape = [batch size, sequence length, 1]
        context_vector = torch.mul(scores, values).sum(dim=-2) #对每一个词的score和对应的value做乘法，然后在seq_len维度上求和，得到context_vector
        # context_vector.shape = [batch size, hidden_dim]
        #socres用于最后的画图
        return context_vector, scores


In [18]:
#把上面的BahdanauAttention写一个例子，看看输出的shape
attention = BahdanauAttention(hidden_dim=1024)
query = torch.randn(2, 1024) #Decoder的隐藏状态 ht
keys = torch.randn(2, 5, 1024) #EO
values = torch.randn(2, 5, 1024) #EO
attn_mask = torch.randint(0, 2, (2, 5))
print(attn_mask)
context_vector, scores = attention(query, keys, values, attn_mask)
print(context_vector.shape)
print(scores.shape)
print(scores)

tensor([[1, 0, 1, 1, 1],
        [0, 0, 1, 0, 0]])
torch.Size([2, 1024])
torch.Size([2, 5, 1])
tensor([[[0.0000],
         [1.0000],
         [0.0000],
         [0.0000],
         [0.0000]],

        [[0.2432],
         [0.2264],
         [0.0000],
         [0.2914],
         [0.2390]]], grad_fn=<SoftmaxBackward0>)


In [19]:
class Decoder(nn.Module):
    def __init__(
        self,
        vocab_size,
        embedding_dim=256,
        hidden_dim=1024,
        num_layers=1,
        ):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.gru = nn.GRU(embedding_dim + hidden_dim, hidden_dim, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size) #最后分类,词典大小是多少，就输出多少个分类
        self.dropout = nn.Dropout(0.6) #0.6可以调整的超参数
        self.attention = BahdanauAttention(hidden_dim) #注意力得到的context_vector

    def forward(self, decoder_input, hidden, encoder_outputs, attn_mask=None):
        #attn_mask是encoder_inputs_mask
        # decoder_input.shape = [batch size, 1]
        assert len(decoder_input.shape) == 2 and decoder_input.shape[-1] == 1, f"decoder_input.shape = {decoder_input.shape} is not valid"
        # hidden.shape = [batch size, hidden_dim]，decoder_hidden,而第一次使用的是encoder的hidden
        assert len(hidden.shape) == 2, f"hidden.shape = {hidden.shape} is not valid"
        # encoder_outputs.shape = [batch size, sequence length, hidden_dim]
        assert len(encoder_outputs.shape) == 3, f"encoder_outputs.shape = {encoder_outputs.shape} is not valid"
        # context_vector.shape = [batch_size, hidden_dim]
        context_vector, attention_score = self.attention(
            query=hidden, keys=encoder_outputs, values=encoder_outputs, attn_mask=attn_mask)
        # decoder_input.shape = [batch size, 1]
        embeds = self.embedding(decoder_input)
        # embeds.shape = [batch size, 1, embedding_dim]
        # context_vector.shape = [batch size, hidden_dim] -->unsqueeze(-2)增加维度 [batch size, 1, hidden_dim]
        embeds = torch.cat([context_vector.unsqueeze(-2), embeds], dim=-1)
        # 新的embeds.shape = [batch size, 1, embedding_dim + hidden_dim]
        seq_output, hidden = self.gru(embeds) #这里需要把前面的decode hidden再次输入，需要改进
        # seq_output.shape = [batch size, 1, hidden_dim]，hidden.shape =[layers,batch size,hidden_dim]
        logits = self.fc(self.dropout(seq_output))
        # logits.shape = [batch size, 1, vocab size]，attention_score = [batch size, sequence length, 1]
        # attention_score是用来画图的
        return logits, hidden, attention_score



In [ ]:
class Sequence2Sequence(nn.Module):
    def __init__(
        self,
        src_vocab_size, #输入词典大小
        trg_vocab_size, #输出词典大小
        encoder_embedding_dim=256,
        encoder_hidden_dim=1024, #encoder_hidden_dim和decoder_hidden_dim必须相同，是因为BahdanauAttention设计的
        encoder_num_layers=1,
        decoder_embedding_dim=256,
        decoder_hidden_dim=1024,
        decoder_num_layers=1,
        bos_idx=1,
        eos_idx=3,
        max_length=512,
        ):
        super().__init__()
        self.bos_idx = bos_idx
        self.eos_idx = eos_idx
        self.max_length = max_length
        self.encoder = Encoder(
            src_vocab_size,
            embedding_dim=encoder_embedding_dim,
            hidden_dim=encoder_hidden_dim,
            num_layers=encoder_num_layers,
            )
        self.decoder = Decoder(
            trg_vocab_size,
            embedding_dim=decoder_embedding_dim,
            hidden_dim=decoder_hidden_dim,
            num_layers=decoder_num_layers,
            )

    def forward(self, *, encoder_inputs, decoder_inputs, attn_mask=None):
        # encoding
        encoder_outputs, hidden = self.encoder(encoder_inputs)
        # decoding with teacher forcing
        bs, seq_len = decoder_inputs.shape
        logits_list = []
        scores_list = []
        for i in range(seq_len):#串行训练
            # 每次迭代生成一个时间步的预测，存储在 logits_list 中，并且记录注意力分数（如果有的话）在 scores_list 中，最后将预测的logits和注意力分数拼接并返回。
            logits, hidden, score = self.decoder(
                decoder_inputs[:, i:i+1], #每次都拿一个时间步
                hidden[-1], #取最后一层的hidden，第一个时间步时，hidden是encoder的hidden，第二个时间步时，hidden是decoder的hidden
                encoder_outputs,
                attn_mask=attn_mask
                )
            logits_list.append(logits) #记录预测的logits，用于计算损失
            scores_list.append(score) #记录注意力分数,用于画图
        #-2维度拼接，是在seq_len维度上拼接，即[batch size, decoder_seq_len, vocab size],scores_list在-1维度拼接，即[batch size, encoder_seq_len, decoder_seq_len]
        return torch.cat(logits_list, dim=-2), torch.cat(scores_list, dim=-1)

    @torch.no_grad() #不计算梯度
    def infer(self, encoder_input, attn_mask=None):
        #infer用于预测
        # encoder_input.shape = [1, sequence length],这只支持batch_size=1
        # encoding
        encoder_outputs, hidden = self.encoder(encoder_input)

        # decoding，[[1]]
        decoder_input = torch.Tensor([self.bos_idx]).reshape(1, 1).to(dtype=torch.int64) #shape为[1,1]，内容为开始标记
        decoder_pred = None
        pred_list = [] #预测序列
        score_list = [] #为了画图，记录注意力分数
        # 从开始标记 bos_idx 开始，迭代地生成序列，直到生成结束标记 eos_idx 或达到最大长度 max_length。
        for _ in range(self.max_length):
            logits, hidden, score = self.decoder(
                decoder_input,
                hidden[-1],
                encoder_outputs,
                attn_mask=attn_mask
                )
            # using greedy search,logits shape = [1, 1, vocab size]
            decoder_pred = logits.argmax(dim=-1)  #最后一维是vocab size，取最大值，得到预测的标量
            decoder_input = decoder_pred
            pred_list.append(decoder_pred.reshape(-1).item()) #decoder_pred从(1,1)变为（1）标量
            score_list.append(score) #记录注意力分数,用于画图

            # stop at eos token
            if decoder_pred == self.eos_idx:
                break

        # return
        return pred_list, torch.cat(score_list, dim=-1)

In [24]:
#做model的前向传播，看看输出的shape
model = Sequence2Sequence(src_vocab_size=len(src_word2idx), trg_vocab_size=len(trg_word2idx))
encoder_inputs = torch.randint(0, 100, (2, 50))
decoder_inputs = torch.randint(0, 100, (2, 60))
attn_mask = torch.randint(0, 2, (2, 50))
logits, scores = model(encoder_inputs=encoder_inputs, decoder_inputs=decoder_inputs, attn_mask=attn_mask)
print(logits.shape)
print(scores.shape)

#计算model的总参数量
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"The model has {count_parameters(model):,} trainable parameters")

torch.Size([2, 60, 11217])
torch.Size([2, 50, 60])
The model has 32,146,386 trainable parameters


## 模型训练

### 损失函数

In [25]:
def cross_entropy_with_padding(logits, labels, padding_mask=None):
    # logits.shape = [batch size, sequence length, num of classes]
    # labels.shape = [batch size, sequence length]
    # padding_mask.shape = [batch size, sequence length] decoder_labels_mask
    bs, seq_len, nc = logits.shape
    loss = F.cross_entropy(logits.reshape(bs * seq_len, nc), labels.reshape(-1), reduction='none') #reduction='none'表示不对batch求平均
    if padding_mask is None:#如果没有padding_mask，就直接求平均
        loss = loss.mean()
    else:
        # 如果提供了 padding_mask，则将padding填充部分的损失去除后计算有效损失的均值。首先，通过将 padding_mask reshape 成一维张量，并取 1 减去得到填充掩码。这样填充部分的掩码值变为 1，非填充部分变为 0。将损失张量与填充掩码相乘，这样填充部分的损失就会变为 0。然后，计算非填充部分的损失和（sum）以及非填充部分的掩码数量（sum）作为有效损失的均值计算。(因为上面我们设计的mask的token是0，所以这里是1-padding_mask)
        padding_mask = 1 - padding_mask.reshape(-1) #将padding_mask reshape成一维张量，mask部分为0，非mask部分为1
        loss = torch.mul(loss, padding_mask).sum() / padding_mask.sum()

    return loss
